# CONCLAVE — Agentic AI Without a Language Model
### Dataset EDA, main working process and a working agent, step by step

**Author:** Aranya Mahmud, Department of Computer Science and Engineering, Daffodil International University
**Code:** [github.com/Aranya69/conclave](https://github.com/Aranya69/conclave) · **Paper:** `paper/main.pdf` in the repository

<a href="https://colab.research.google.com/github/Aranya69/conclave/blob/main/notebooks/CONCLAVE_Colab_Demo.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

CONCLAVE is an agentic AI framework in which a **committee of classical machine learning models** replaces the large language model (LLM) as the decision-making core. The agent therefore works **without the internet** and on **low-cost hardware** such as a Raspberry Pi, which matters in disaster response, where the network is often the first thing to fail.

This notebook reproduces the main working process of the paper on a free Colab CPU (no GPU, no API key). It takes about **10 minutes** with *Runtime → Run all*.

| Part | Content |
|---|---|
| 0 | Setup: clone the code, download the two public datasets |
| 1 | Exploratory data analysis (EDA): HumAID, BFCL v3 and the flood-response simulator |
| 2 | Main working process: features → committee → uncertainty → gate → agent loop |
| 3 | Tool routing (BFCL) and crisis triage (HumAID) |
| 4 | Experiments: agent behaviour without internet, comparison with other ML agents, full paper results |
| 5 | Edge deployment: the NumPy-only runtime for low-cost boards |

## 0. Setup
The cell below clones the repository, installs the few missing packages and downloads the two public datasets (BFCL v3 and HumAID, about 19 MB) from their original sources. The flood-response data are generated by the simulator in the repository, so they need no download.

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/conclave"):
        !git clone -q https://github.com/Aranya69/conclave.git /content/conclave
    %cd /content/conclave
    !pip install -q lightgbm psutil pyarrow
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                      # running from a local clone
sys.path.insert(0, os.getcwd())
!{sys.executable} scripts/fetch_data.py

In [ ]:
import warnings, time, json, re, collections, subprocess, textwrap
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, lightgbm
from IPython.display import display, Image, Markdown

from conclave import Agent, EnsembleBrain, EpisodicMemory, UncertaintyGate
from conclave.brain.featurize import BudgetedTextFeaturizer, FusedFeaturizer, TabularFeaturizer
from conclave.brain.uncertainty import decompose, decompose_batch
from conclave.brain.router import ToolRouter
from conclave.core.types import Action, ActionKind, Observation
from conclave.envs.disaster import ACTIONS, DisasterConfig, FloodResponseEnv, expert_policy
from conclave.planning.rules import ConstraintSet, require_confidence
from experiments.exp1_tool_routing import load_pairs, load_irrelevance
from experiments.exp3_sequential import collect, run_policy, train_conclave, static_rules

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

def boxplot(ax, data, labels, horizontal=False):
    # boxplot that works across matplotlib versions
    try:
        ax.boxplot(data, orientation="horizontal" if horizontal else "vertical", showfliers=False)
    except TypeError:                       # matplotlib < 3.10
        ax.boxplot(data, vert=not horizontal, showfliers=False)
    ticks = range(1, len(labels) + 1)
    if horizontal:
        ax.set_yticks(ticks); ax.set_yticklabels(labels)
    else:
        ax.set_xticks(ticks); ax.set_xticklabels(labels, rotation=25)
pd.set_option("display.width", 160)
SEED = 42
np.random.seed(SEED)

# Experiment sizes. The paper uses the values in brackets; the defaults keep the
# whole notebook at about 10 minutes on a free Colab CPU.
N_TRAIN_EPISODES = 160   # [160] expert episodes for behaviour cloning
N_EVAL_EPISODES  = 40    # [120] evaluation episodes per condition (Section 4.1)
N_COMPARE_EPISODES = 120 # [120] episodes for the comparison with other agents (Section 4.2)
HUMAID_BITS      = 12    # [12, 14, 16] hash buckets = 2**bits per text view
FAST_TEST = os.environ.get("CONCLAVE_FAST_TEST") == "1"   # only for testing this notebook
if FAST_TEST:
    N_TRAIN_EPISODES, N_EVAL_EPISODES, N_COMPARE_EPISODES, HUMAID_BITS = 40, 6, 6, 10

print(f"python {sys.version.split()[0]} | numpy {np.__version__} | scikit-learn {sklearn.__version__}"
      f" | lightgbm {lightgbm.__version__} | pandas {pd.__version__}")

## 1. Exploratory data analysis
Four test beds are used in the paper. Two are public datasets (**HumAID** and **BFCL v3**), and two were built for the study (the **flood-response simulator** and the **edge hardware profiles**). This part examines the three that contain data.

### 1.1 HumAID — crisis tweets
HumAID contains tweets posted during real disasters, each labelled with one of ten humanitarian categories. The agent must sort every incoming message into its category (triage). We keep the authors' official train/validation/test split.

In [ ]:
tr = pd.read_parquet("data/humaid/train.parquet")
va = pd.read_parquet("data/humaid/validation.parquet")
te = pd.read_parquet("data/humaid/test.parquet")
if FAST_TEST:
    tr, va, te = tr.sample(4000, random_state=0), va.sample(1000, random_state=0), te.sample(2000, random_state=0)
humaid = pd.concat([tr.assign(split="train"), va.assign(split="validation"), te.assign(split="test")],
                   ignore_index=True)
print(f"tweets: {len(humaid):,}   train {len(tr):,} | validation {len(va):,} | test {len(te):,}")
print("missing values:", humaid[["tweet_text", "class_label"]].isna().sum().to_dict())
print("duplicate tweet texts:", int(humaid.tweet_text.duplicated().sum()))
display(humaid.sample(5, random_state=1)[["tweet_text", "class_label", "split"]])

In [ ]:
dist = pd.crosstab(humaid.class_label, humaid.split)[["train", "validation", "test"]]
dist["total"] = dist.sum(axis=1)
dist["share (%)"] = (100 * dist.total / dist.total.sum()).round(1)
dist = dist.sort_values("total", ascending=False)
display(dist)
print(f"imbalance ratio (largest / smallest class): {dist.total.max() / dist.total.min():.1f} : 1")

In [ ]:
humaid["words"] = humaid.tweet_text.str.split().str.len()
humaid["chars"] = humaid.tweet_text.str.len()
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
dist.total.iloc[::-1].plot.barh(ax=ax[0], color="#1b6ca8")
ax[0].set_title("(a) Class distribution"); ax[0].set_xlabel("tweets")
ax[1].hist(humaid.words, bins=40, color="#5dade2", edgecolor="white")
ax[1].set_title("(b) Tweet length"); ax[1].set_xlabel("words per tweet")
order = dist.index.tolist()
boxplot(ax[2], [humaid.loc[humaid.class_label == c, "words"] for c in order],
        [c.replace("_", " ")[:28] for c in order], horizontal=True)
ax[2].set_title("(c) Length by class"); ax[2].set_xlabel("words per tweet")
plt.tight_layout(); plt.show()
print(humaid[["words", "chars"]].describe().round(1).T)

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
stop = set(ENGLISH_STOP_WORDS) | {"rt", "amp", "https", "http", "co", "via", "just", "like"}
def top_words(texts, k=8):
    c = collections.Counter()
    for t in texts:
        c.update(w for w in re.findall(r"[a-z]{3,}", t.lower()) if w not in stop)
    return ", ".join(w for w, _ in c.most_common(k))
top = pd.DataFrame({"most frequent words": {c: top_words(tr.loc[tr.class_label == c, "tweet_text"]) for c in order}})
display(top)

**Observations.** The classes are strongly imbalanced (rescue, volunteering or donation effort is by far the largest; missing or found people is the smallest), so the paper reports both weighted and macro F1. Tweets are short (about 20 words on average), and each class has a distinctive vocabulary, which is why hashed word and character n-grams work well for this task.

### 1.2 BFCL v3 — tool-calling requests
The Berkeley Function Calling Leaderboard (BFCL v3) pairs each user request with the specifications of several candidate tools (functions). The agent must choose the tool that serves the request, or recognise that no tool fits (abstention).

In [ ]:
stats = []
for name in ["BFCL_v3_simple", "BFCL_v3_multiple", "BFCL_v3_live_simple", "BFCL_v3_live_multiple"]:
    d = load_pairs(name)
    stats.append({"split": name.replace("BFCL_v3_", ""), "requests": len(d),
                  "candidate tools (mean)": round(np.mean([len(x["funcs"]) for x in d]), 2),
                  "candidate tools (max)": max(len(x["funcs"]) for x in d),
                  "request words (mean)": round(np.mean([len(x["q"].split()) for x in d]), 1)})
for name in ["BFCL_v3_irrelevance", "BFCL_v3_live_irrelevance"]:
    d = load_irrelevance(name)
    stats.append({"split": name.replace("BFCL_v3_", "") + " (no tool fits)", "requests": len(d),
                  "candidate tools (mean)": round(np.mean([len(x["funcs"]) for x in d]), 2),
                  "candidate tools (max)": max(len(x["funcs"]) for x in d),
                  "request words (mean)": round(np.mean([len(x["q"].split()) for x in d]), 1)})
display(pd.DataFrame(stats).set_index("split"))

routing = [x for n in ["BFCL_v3_multiple", "BFCL_v3_live_multiple"] for x in load_pairs(n) if len(x["funcs"]) > 1]
print(f"routing requests used in the paper (>= 2 candidate tools): {len(routing):,}")

In [ ]:
live = [x for x in load_pairs("BFCL_v3_live_multiple") if len(x["funcs"]) > 1]
n_params = [len((f.get("parameters") or {}).get("properties") or {}) for x in live for f in x["funcs"]]
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ax[0].hist([len(x["funcs"]) for x in live], bins=range(2, 40), color="#1b6ca8", edgecolor="white")
ax[0].set_title("(a) Candidate tools per request"); ax[0].set_xlabel("tools")
ax[1].hist([min(len(x["q"].split()), 100) for x in live], bins=40, color="#5dade2", edgecolor="white")
ax[1].set_title("(b) Request length (capped at 100 words)"); ax[1].set_xlabel("words")
ax[2].hist(n_params, bins=range(0, 16), color="#16a085", edgecolor="white")
ax[2].set_title("(c) Parameters per tool"); ax[2].set_xlabel("parameters")
plt.tight_layout(); plt.show()

ex = live[3]
print("REQUEST:", textwrap.shorten(ex["q"], 220))
print("GOLD TOOL:", ex["target"])
for f in ex["funcs"]:
    print(f"  - {f['name']:<40s} {textwrap.shorten(str(f.get('description', '')), 90)}")

### 1.3 Flood-response simulator — sequential decisions
Classification datasets cannot show whether a framework behaves well as an *agent*, because they contain no sequence of decisions. The simulator operates a river district for 40 steps. At each step the agent sees 12 noisy sensor readings and a short field report, and chooses one of six tools: `monitor`, `issue_alert`, `evacuate_zone`, `deploy_rescue`, `request_supplies` and `open_sluice` (irreversible). The training labels come from an expert controller that can see the hidden flood risk, which the agent never observes (behaviour cloning).

In [ ]:
t0 = time.time()
obs, lab = collect(N_TRAIN_EPISODES, seed0=1000)          # same corpus as the paper
flood = pd.DataFrame([o.features for o in obs])
flood["true_risk"] = [o.meta["true_risk"] for o in obs]
flood["report"] = [o.text for o in obs]
flood["action"] = lab
print(f"{N_TRAIN_EPISODES} expert episodes -> {len(flood):,} labelled decisions, "
      f"{flood.shape[1] - 3} sensor features ({time.time() - t0:.1f}s)")
display(flood.drop(columns=["report"]).describe().T.round(2))
display(flood.sample(4, random_state=3)[["river_level", "rainfall", "stranded", "true_risk", "report", "action"]])

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
flood.action.value_counts().plot.bar(ax=ax[0], color="#1b6ca8", rot=25)
ax[0].set_title("(a) Expert action distribution"); ax[0].set_ylabel("decisions")
acts = flood.action.value_counts().index.tolist()
boxplot(ax[1], [flood.loc[flood.action == a, "river_level"] for a in acts], acts)
ax[1].set_title("(b) River level by action"); ax[1].set_ylabel("m")
boxplot(ax[2], [flood.loc[flood.action == a, "true_risk"] for a in acts], acts)
ax[2].set_title("(c) Hidden flood risk by action")
plt.tight_layout(); plt.show()

num = flood.drop(columns=["report", "action"])
num = num.loc[:, num.std() > 0]                 # constant columns have no correlation
corr = num.corr()
fig, ax = plt.subplots(figsize=(8.5, 7))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=60, ha="right", fontsize=8)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns, fontsize=8)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.1f}", ha="center", va="center", fontsize=6)
ax.grid(False); fig.colorbar(im, ax=ax, shrink=0.8); ax.set_title("(d) Correlation of sensor features")
plt.tight_layout(); plt.show()

In [ ]:
# one episode, with the network failing at random, to show the dynamics
env = FloodResponseEnv(DisasterConfig(seed=2024, horizon=40, outage_prob=0.3))
o = env.reset(); rng = np.random.default_rng(0); traj = []
for t in range(40):
    a = expert_policy(o, rng)
    traj.append({"t": t, "level": o.features["river_level"], "rain": o.features["rainfall"],
                 "risk": o.meta["true_risk"], "online": o.meta["online"], "action": a, "report": o.text})
    o, r, done, _ = env.step(Action(kind=ActionKind.TOOL, tool=a))
    if done:
        break
traj = pd.DataFrame(traj)
fig, (ax, axa) = plt.subplots(2, 1, figsize=(14, 6.2), sharex=True, gridspec_kw={"height_ratios": [2.2, 1]})
ax.plot(traj.t, traj.level, "-o", ms=3, label="river level (m)", color="#1b6ca8")
ax.plot(traj.t, traj.rain, "-", lw=1, label="rainfall", color="#7f8c8d")
ax2 = ax.twinx(); ax2.plot(traj.t, traj.risk, "--", color="#c0392b", label="hidden flood risk"); ax2.set_ylim(0, 1.05)
ax2.grid(False); ax2.set_ylabel("risk")
for a_ in (ax, axa):
    for t in traj.t[~traj.online]:
        a_.axvspan(t - 0.5, t + 0.5, color="#f5b7b1", alpha=0.35, lw=0)
ax.legend(loc="upper left"); ax2.legend(loc="upper right")
ax.set_title("One simulated flood episode (red bands: network down)")
acts_order = ["monitor", "issue_alert", "evacuate_zone", "deploy_rescue", "request_supplies", "open_sluice"]
axa.scatter(traj.t, [acts_order.index(a) for a in traj.action], c="#1b6ca8", s=28)
axa.set_yticks(range(len(acts_order))); axa.set_yticklabels(acts_order, fontsize=8)
axa.set_xlabel("time step"); axa.set_title("expert action at each step", fontsize=9)
plt.tight_layout(); plt.show()
print("Example field reports by risk band:")
for lo, hi in [(0, 0.2), (0.2, 0.45), (0.45, 0.75), (0.75, 1.01)]:
    s = flood[(flood.true_risk >= lo) & (flood.true_risk < hi)].report
    print(f"  risk {lo:.2f}-{hi:.2f}: {s.iloc[0] if len(s) else '-'}")

**Observations.** The expert mostly monitors and acts only when the hidden risk rises, so the action classes are imbalanced. River level, rainfall and the field-report wording all carry information about the risk, but none of them alone determines the action, which is why the agent combines sensor features and text. The red bands show how often a cloud-dependent agent would be cut off.

**Data quality check.** Before choosing these test beds we audited a widely used Bangladesh flood-prediction dataset: a depth-1 decision stump on rainfall alone scored 98.4% because 92% of rows were positive and the label was essentially a rainfall threshold. Such a benchmark cannot separate a good model from a trivial one, so it was not used.

## 2. Main working process
The figure below (Fig. 2 of the paper) shows what the agent does at every time step. The cells after it run each stage on real data.

In [ ]:
display(Image("paper/figures/fig_method.png", width=1000))
display(Image("paper/figures/fig_agent.png", width=520))

### 2.1 Feature engineering: hashed TF–IDF text features + standardised sensors
A report is split into word 1–2-grams and character 3–5-grams. Instead of storing a vocabulary, every n-gram is mapped to a column by a hash function, $j(g)=|h(g)| \bmod D$, and the counts are re-weighted by sublinear TF–IDF, $\tilde u_j=(1+\ln c_j)\,\mathrm{idf}_j$, then normalised to unit length. Sensor readings are standardised, $z_i=(s_i-\mu_i)/\sigma_i$. The featuriser therefore has a **fixed size**, independent of the amount of text seen.

In [ ]:
from conclave.edge.runtime import word_ngrams, _bucket
fz = FusedFeaturizer(text=BudgetedTextFeaturizer(n_word_features=2**13, n_char_features=2**13),
                     tabular=TabularFeaturizer()).fit(obs)
X = fz.transform(obs)
print(f"feature matrix: {X.shape[0]:,} decisions x {X.shape[1]:,} features "
      f"(12 sensors + 2^13 word buckets + 2^13 char buckets); "
      f"{X.getnnz(axis=1).mean():.0f} non-zeros per row on average")

o = obs[150]
print("\nreport:", o.text)
xw = fz.text.transform([o.text])[:, :2**13]
grams = word_ngrams(o.text, (1, 2), True)
rows = [{"word n-gram": g, "hash bucket j": _bucket(g, 2**13), "TF-IDF weight": round(xw[0, _bucket(g, 2**13)], 3)}
        for g in dict.fromkeys(grams)]
display(pd.DataFrame(rows).head(10))
z = fz.tabular.transform([o.features])[0]
display(pd.DataFrame({"raw": [o.features[k] for k in fz.tabular.keys_], "z-score": z.round(2)},
                     index=fz.tabular.keys_).T.round(2))

### 2.2 The committee: logistic regression + SGD (modified Huber) + naive Bayes
Each member gives a posterior over the six actions, and the committee averages them (soft voting), $\bar p(a\mid x)=\sum_m w_m\,p_m(a\mid x)$ with $w_m=1/M$. The members are deliberately different (discriminative, margin-based and generative), so that their disagreement is informative. Note that one decision at a time, the logistic-regression member alone is slightly more accurate than the committee; the committee's benefits appear in its uncertainty estimate (Section 2.3) and in closed-loop control, where errors compound (Section 4.2).

In [ ]:
te_obs, te_lab = collect(60 if not FAST_TEST else 10, seed0=60000)    # unseen episodes
Xte = fz.transform(te_obs)
t0 = time.time()
brain = EnsembleBrain(weighting="uniform").fit(X, lab)
print(f"trained in {time.time() - t0:.1f}s on {X.shape[0]:,} decisions; members: {brain.member_names_}")
M = brain.member_proba(Xte)                                   # (members, samples, actions)
acc = {name: (brain.classes_[M[i].argmax(1)] == te_lab).mean() for i, name in enumerate(brain.member_names_)}
acc["committee (soft vote)"] = (brain.predict(Xte) == te_lab).mean()
display(pd.DataFrame({"one-step accuracy (%)": {k: round(100 * v, 2) for k, v in acc.items()}}))

from sklearn.metrics import ConfusionMatrixDisplay
fig, ax = plt.subplots(figsize=(6.5, 5.5))
ConfusionMatrixDisplay.from_predictions(te_lab, brain.predict(Xte), ax=ax, cmap="Blues",
                                        xticks_rotation=35, colorbar=False)
ax.set_title("Committee on held-out expert decisions"); ax.grid(False); plt.tight_layout(); plt.show()

### 2.3 Uncertainty decomposition: aleatoric vs epistemic
With $\tilde p=\frac1M\sum_m p_m$ and entropy $H$, the total uncertainty splits into an **aleatoric** part (the members agree that the case is ambiguous) and an **epistemic** part (the members disagree, i.e. the agent is outside its competence):

$$\frac{H[\tilde p]}{\ln K}=\underbrace{\frac{1}{M\ln K}\sum_m H[p_m]}_{U^{ale}}+\underbrace{\Big(\frac{H[\tilde p]}{\ln K}-\frac{1}{M\ln K}\sum_m H[p_m]\Big)}_{U^{epi}}$$

Below, three decisions are shown member by member, and then the sensors are degraded (60% of readings dropped) to show that epistemic uncertainty rises when the inputs become unfamiliar.

In [ ]:
from experiments.exp6_ablation import perturb
stats = decompose_batch(M)
picks = [int(np.argmin(stats["total"])), int(np.argsort(stats["aleatoric"])[-1]), int(np.argsort(stats["epistemic"])[-1])]
for i, why in zip(picks, ["clear case", "most ambiguous case", "largest disagreement"]):
    tab = pd.DataFrame(M[:, i, :], index=brain.member_names_, columns=brain.classes_).round(2)
    s = decompose(M[:, i, :])
    print(f"\n{why}: expert action = {te_lab[i]} | confidence {s.confidence:.2f} | "
          f"aleatoric {s.aleatoric:.2f} | epistemic {s.epistemic:.2f}")
    display(tab)

from sklearn.metrics import roc_auc_score
Xsh = fz.transform(perturb(te_obs, dropout=0.6))
Ms = brain.member_proba(Xsh); ss = decompose_batch(Ms)
lab_shift = np.r_[np.zeros(len(te_lab)), np.ones(len(te_lab))]
print(f"\naccuracy clean {100 * (brain.predict(Xte) == te_lab).mean():.1f}% -> with 60% sensor dropout "
      f"{100 * (brain.classes_[Ms.mean(0).argmax(1)] == te_lab).mean():.1f}%")
print(f"AUROC for detecting the degraded inputs: epistemic {roc_auc_score(lab_shift, np.r_[stats['epistemic'], ss['epistemic']]):.3f}"
      f" | confidence {roc_auc_score(lab_shift, np.r_[-stats['confidence'], -ss['confidence']]):.3f}")
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.hist(stats["epistemic"], bins=40, alpha=0.7, label="clean sensors", color="#1b6ca8")
ax.hist(ss["epistemic"], bins=40, alpha=0.6, label="60% sensors dropped", color="#c0392b")
ax.set_xlabel("epistemic uncertainty"); ax.set_ylabel("decisions"); ax.legend(); plt.tight_layout(); plt.show()

### 2.4 The uncertainty gate
The gate maps the uncertainty to one of four control actions: **act**, **gather** more evidence, **abstain**, or **escalate** to a human. Its confidence and epistemic thresholds are not chosen by hand: they are calibrated on held-out data so that the decisions the agent acts on reach a target precision while the agent acts as often as possible (selective prediction).

In [ ]:
va_obs, va_lab = collect(30 if not FAST_TEST else 6, seed0=50000)
gate = UncertaintyGate(allow_gather=False)
gate.calibrate(brain.member_proba(fz.transform(va_obs)), np.searchsorted(brain.classes_, va_lab),
               target_precision=0.99)
c = gate.calibration_
print(f"calibrated thresholds: confidence >= {gate.confidence_min:.3f}, epistemic <= {gate.epistemic_max:.3f}")
if c["achieved"]:
    print(f"on validation: precision {100 * c['achieved']['precision']:.1f}% "
          f"at coverage {100 * c['achieved']['coverage']:.1f}%")
out = pd.DataFrame({
    "clean sensors": pd.Series([gate(decompose(M[:, i, :])).value for i in range(M.shape[1])]).value_counts(),
    "60% sensors dropped": pd.Series([gate(decompose(Ms[:, i, :])).value for i in range(Ms.shape[1])]).value_counts()}).fillna(0).astype(int)
display(out)

### 2.5 The agent loop in action: a flood episode while the network fails
The complete agent combines the committee, the gate, an episodic memory (case-based reasoning) and a safety constraint: the irreversible `open_sluice` tool may fire only when the committee is at least 60% confident. Every decision is made locally; the column `net` shows when a cloud-dependent agent would have been cut off.

When the gate **escalates**, the decision is handed to a local human operator (for example a disaster-management officer on site). In the simulation the operator is represented by the expert controller, who can see the real river. The column `decided by` shows how rarely the agent needs help.

In [ ]:
loop_gate = UncertaintyGate(epistemic_max=0.35, aleatoric_max=0.80, confidence_min=0.25,
                            margin_min=0.02, allow_gather=False)
agent = Agent(brain=brain, featurizer=fz, gate=loop_gate, memory=EpisodicMemory(capacity=500),
              constraints=ConstraintSet([require_confidence(0.6, ["open_sluice"])]))
env = FloodResponseEnv(DisasterConfig(seed=2024, horizon=40, outage_prob=0.3))
o = env.reset(); rows = []; total = 0.0; operator = np.random.default_rng(0)
for t in range(40):
    d = agent.decide(o)
    act, who = d.action, "agent"
    if act.kind is ActionKind.ESCALATE:          # hand this one decision to the human operator
        act, who = Action(kind=ActionKind.TOOL, tool=expert_policy(o, operator)), "human"
    rows.append({"t": t, "level": round(o.features["river_level"], 2), "risk": round(o.meta["true_risk"], 2),
                 "net": "up" if o.meta["online"] else "DOWN", "gate": d.gate,
                 "action": act.tool, "decided by": who, "confidence": round(d.confidence, 2),
                 "epistemic": round(d.disagreement, 3), "ms": round(d.latency_ms, 2)})
    o, r, done, _ = env.step(act); total += r
    if done:
        break
episode = pd.DataFrame(rows)
display(episode)
print(f"episode return {total:.1f} | {env.summary()}")
print(f"network down on {(episode.net == 'DOWN').sum()} of {len(episode)} steps -- the agent still decided locally on every step")
print(f"decisions handed to the human operator: {(episode['decided by'] == 'human').sum()} of {len(episode)}")
print(f"median decision time: {episode.ms.median():.2f} ms")

## 3. Tool routing and crisis triage
### 3.1 Tool routing on BFCL v3 (no language model)
Instead of asking an LLM to name a tool, CONCLAVE scores every (request, tool) pair with a ranking committee (LightGBM + random forest + logistic regression) over 11 pairwise features, and picks the highest score. Because it scores *pairs*, a tool registered after training can be chosen without retraining.

In [ ]:
from sklearn.model_selection import train_test_split
pairs = [x for x in load_pairs("BFCL_v3_multiple" if FAST_TEST else "BFCL_v3_live_multiple") if len(x["funcs"]) > 1]
tr_p, te_p = train_test_split(pairs, test_size=0.3, random_state=42)
t0 = time.time()
router = ToolRouter(abstain=False).fit([x["q"] for x in tr_p], [x["funcs"] for x in tr_p], [x["target"] for x in tr_p])
print(f"router trained on {len(tr_p)} requests in {time.time() - t0:.1f}s")
hits, lexical, lat = [], [], []
for x in te_p:
    r = router.route(x["q"], x["funcs"]); hits.append(r.tool == x["target"]); lat.append(r.latency_ms)
    F = np.asarray([router._pair_features(x["q"], f) for f in x["funcs"]])
    lexical.append(x["funcs"][int(np.argmax(F[:, 0]))]["name"] == x["target"])
print(f"top-1 accuracy on {len(te_p)} unseen requests: CONCLAVE {100 * np.mean(hits):.1f}% | "
      f"lexical similarity only {100 * np.mean(lexical):.1f}% | {np.mean(lat):.2f} ms per decision")

In [ ]:
# open vocabulary: five tools that never appeared in training
new_tools = [
    {"name": "get_weather", "description": "Current weather forecast for a city",
     "parameters": {"properties": {"city": {"type": "string", "description": "City name"}}, "required": ["city"]}},
    {"name": "send_sms_alert", "description": "Broadcast an emergency SMS warning to a zone",
     "parameters": {"properties": {"zone": {"type": "string", "description": "Zone id"},
                                   "message": {"type": "string", "description": "Alert text"}}, "required": ["zone", "message"]}},
    {"name": "dispatch_rescue_boat", "description": "Send a rescue boat to people trapped by flood water",
     "parameters": {"properties": {"location": {"type": "string", "description": "Where people are stranded"}}, "required": ["location"]}},
    {"name": "check_river_gauge", "description": "Read the latest water level from a river gauge station",
     "parameters": {"properties": {"station": {"type": "string", "description": "Gauge station id"}}, "required": ["station"]}},
    {"name": "order_food_supplies", "description": "Order food and drinking water for a relief camp",
     "parameters": {"properties": {"camp": {"type": "string", "description": "Relief camp name"},
                                   "quantity": {"type": "integer", "description": "Number of packs"}}, "required": ["camp"]}},
]
for q in ["there are people stuck on a roof in sector 4, send help",
          "what is the water level at the Sirajganj station right now",
          "warn everyone in zone 7 that the embankment may break",
          "the shelter at Kurigram needs 300 packs of food and water"]:
    r = router.route(q, new_tools)
    best = sorted(r.scores.items(), key=lambda kv: -kv[1])[:3]
    print(f"{q[:60]:<60s} -> {r.tool:<20s} {', '.join(f'{k} {v:.2f}' for k, v in best)}")

### 3.2 Crisis triage on HumAID
The same decision committee (logistic regression + SGD + naive Bayes) classifies tweets on hashed TF–IDF features. The number of hash buckets is the deployment parameter: the paper uses $2^{12}$, $2^{14}$ and $2^{16}$; this notebook uses $2^{12}$ to stay fast.

In [ ]:
from sklearn.metrics import f1_score, classification_report
t0 = time.time()
fzh = BudgetedTextFeaturizer(n_word_features=2**HUMAID_BITS, n_char_features=2**HUMAID_BITS)
Xtr_h = fzh.fit(tr.tweet_text.tolist()).transform(tr.tweet_text.tolist())
Xva_h, Xte_h = fzh.transform(va.tweet_text.tolist()), fzh.transform(te.tweet_text.tolist())
brain_h = EnsembleBrain(weighting="uniform").fit(Xtr_h, tr.class_label.values)
pred_h = brain_h.predict(Xte_h)
wf1 = f1_score(te.class_label, pred_h, average="weighted"); mf1 = f1_score(te.class_label, pred_h, average="macro")
print(f"2^{HUMAID_BITS} buckets: weighted F1 {wf1:.3f} | macro F1 {mf1:.3f} | {time.time() - t0:.0f}s to featurise and train")
print(classification_report(te.class_label, pred_h, digits=3))
display(pd.DataFrame({"weighted F1": {
    "RoBERTa (reported by the HumAID authors)": 0.760, "FastText (reported)": 0.744,
    "SVM (reported)": 0.731, "Random forest (reported)": 0.707,
    f"CONCLAVE, 2^{HUMAID_BITS} buckets (this run)": round(wf1, 3),
    "CONCLAVE, 2^16 buckets (paper)": 0.743}}))
g = UncertaintyGate()
g.calibrate(brain_h.member_proba(Xva_h), np.searchsorted(brain_h.classes_, va.class_label.values), target_precision=0.90)
if g.calibration_["achieved"]:
    print(f"calibrated gate: {100 * g.calibration_['achieved']['coverage']:.1f}% of tweets triaged automatically at "
          f"{100 * g.calibration_['achieved']['precision']:.1f}% precision; the rest go to a human analyst")

## 4. Experiments
### 4.1 Without internet: CONCLAVE vs a cloud-dependent agent
The cloud agent follows the **expert** policy itself (it sees the hidden risk), so it is the upper bound of a remote LLM-based agent. It cannot act while the network is down; a cache variant repeats its last action. CONCLAVE was trained by behaviour cloning on the same expert, so any advantage comes from availability alone.

In [ ]:
t0 = time.time()
brain_c, fz_c, _ = train_conclave()              # the exact agent of the paper (160 episodes)
grid = [0.0, 0.1, 0.3, 0.5]
names = {"conclave": "CONCLAVE (local)", "cloud_oracle": "cloud agent", "cloud_oracle_cache": "cloud agent + cache",
         "static_rules": "static threshold rules"}
res = []
for q in grid:
    for pol in names:
        r = run_policy(pol, q, N_EVAL_EPISODES, brain_c, fz_c, None)
        res.append({"policy": names[pol], "offline (%)": round(100 * r["measured_offline_fraction"], 1),
                    "return": round(r["mean_return"], 1), "casualties": round(r["mean_casualties"], 1)})
res = pd.DataFrame(res)
display(res.pivot(index="offline (%)", columns="policy", values="return"))
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for pol, g in res.groupby("policy"):
    ax[0].plot(g["offline (%)"], g["return"], "-o", label=pol)
    ax[1].plot(g["offline (%)"], g["casualties"], "-o", label=pol)
ax[0].set_xlabel("steps with no connectivity (%)"); ax[0].set_ylabel("mean return"); ax[0].legend()
ax[1].set_xlabel("steps with no connectivity (%)"); ax[1].set_ylabel("mean casualties")
plt.tight_layout(); plt.show()
print(f"({N_EVAL_EPISODES} episodes per point, {time.time() - t0:.0f}s; the paper uses 120)")

### 4.2 Comparison with other machine-learning-based agents
All agents below run offline on the same CPU and are trained on the same decisions and features; only the decision core differs. They are tested with the network down for about 73% of the steps (no internet) and on cost (latency, model size, estimated latency on a Raspberry Pi Zero 2 W). The same 120 episode seeds as the paper are used, so the returns reproduce the "73% offline" column of Table IX. This cell takes about five minutes, because the case-based ($k$-NN) and hard-vote agents are slow.

In [ ]:
import io, joblib
from experiments.exp7_ml_agents import flood_agents
from conclave.edge.profile import PROFILES
tr_obs7, tr_lab7 = collect(N_TRAIN_EPISODES, seed0=1000)
Xtr7, Xte7 = fz_c.transform(tr_obs7), fz_c.transform(te_obs)
agents = {k: v for k, v in flood_agents().items() if "log-loss" not in k}
if FAST_TEST:
    agents = {k: agents[k] for k in ["Decision tree", "Logistic regression", "LightGBM"]}
agents["CONCLAVE (ours)"] = brain_c
rows = []
for name, b in agents.items():
    t0 = time.time()
    if name != "CONCLAVE (ours)":
        b.fit(Xtr7, tr_lab7)
    r = run_policy("conclave", 0.5, N_COMPARE_EPISODES, b, fz_c, None)
    buf = io.BytesIO(); joblib.dump({"fz": fz_c, "brain": b}, buf, compress=3)
    rows.append({"agent": name, "one-step acc (%)": round(100 * (b.predict(Xte7) == te_lab).mean(), 1),
                 "return, 73% offline": round(r["mean_return"], 1),
                 "casualties": round(r["mean_casualties"], 1),
                 "latency (ms)": round(r["median_decision_ms"], 2),
                 "est. Pi Zero 2 W (ms)": round(r["median_decision_ms"] / PROFILES["rpi_zero2w"].scalar_score, 1),
                 "model (KB)": round(buf.tell() / 1e3)})
    print(f"  {name:<22s} done in {time.time() - t0:5.1f}s")
cmp = pd.DataFrame(rows).set_index("agent").sort_values("return, 73% offline", ascending=False)
display(cmp)
ax = cmp["return, 73% offline"].iloc[::-1].plot.barh(figsize=(8, 4.2),
        color=["#1b6ca8" if "CONCLAVE" in i else "#bdc3c7" for i in cmp.index[::-1]])
ax.set_xlabel("mean episode return with the network down (73% of steps)"); plt.tight_layout(); plt.show()

**Reading the comparison.** Several single models imitate the expert slightly more accurately one decision at a time, but the committee obtains the highest return and the fewest casualties when the decisions are chained over a whole episode: its members make less correlated errors, so its policy drifts less. Small differences from the paper can appear if Colab's library versions differ from those used for the paper; the paper's own result files, with paired bootstrap tests, are loaded below.

### 4.3 Full results reported in the paper
These tables and figures are read from the result files in `results/`, which were produced by the experiment scripts in `experiments/` (each records its platform, seed and code version).

In [ ]:
r7 = json.load(open("results/exp7_ml_agents.json"))["flood"]
full = pd.DataFrame([{"agent": r["agent"], "one-step acc (%)": round(100 * r["accuracy"], 1),
                      "return, network up": round(r["return_offline0"], 1),
                      "return, 73% offline": round(r["return_offline73"], 1),
                      "casualties, 73% offline": round(r["casualties_offline73"], 1),
                      "latency (ms)": round(r["median_decision_ms"], 2),
                      "est. Pi Zero 2 W (ms)": round(r["est_ms_pi_zero2w"], 1),
                      "model (KB)": round(r["size_kb"]), "peak RAM (MB)": round(r["peak_rss_mb"])}
                     for r in r7["rows"] if "log-loss" not in r["agent"]]).set_index("agent")
display(Markdown("**Table IX of the paper: CONCLAVE against eight other ML agents (120 episodes)**"))
display(full.sort_values("return, 73% offline", ascending=False))
sig = [t for t in r7["paired_tests"] if t["outage_prob"] == 0.5 and "log-loss" not in t["vs"]]
print("paired bootstrap, CONCLAVE minus each agent (73% offline):")
for t in sig:
    print(f"  vs {t['vs']:<22s} {t['delta_mean']:+6.1f}  95% CI [{t['ci_low']:+.1f}, {t['ci_high']:+.1f}]"
          f"  {'significant' if t['significant'] else 'not significant'}")
for f in ["fig1_availability.png", "fig6_ml_agents.png", "fig5_routing.png", "fig2_frontier.png"]:
    display(Image(f"paper/figures/{f}", width=850))

## 5. Edge deployment: the NumPy-only runtime
Training needs scikit-learn and SciPy, but deciding does not: every member of the committee is linear in the features. The trained agent is exported to one small `.npz` file, and `conclave/edge/runtime.py` (a single file that imports only NumPy) reproduces every decision exactly. This is what makes the agent fit comfortably on a Raspberry Pi Zero 2 W (512 MB RAM).

In [ ]:
from conclave.edge.runtime import export_agent, EdgeAgent
info = export_agent("conclave_agent.npz", fz_c, brain_c, loop_gate)
edge = EdgeAgent("conclave_agent.npz")
print(f"exported file: {os.path.getsize('conclave_agent.npz') / 1e3:.0f} KB "
      f"({info['columns_kept']:,} of {info['columns_total']:,} feature columns kept)")

reps = brain_c.report(fz_c.transform(te_obs))
same = sum(edge.decide(o.features, o.text)["action"] == str(rp.argmax) for o, rp in zip(te_obs, reps))
diff = max(float(np.abs(edge.decide(o.features, o.text)["posterior"] - rp.posterior).max())
           for o, rp in zip(te_obs[:300], reps[:300]))
print(f"identical actions: {same}/{len(te_obs)} | largest posterior difference: {diff:.1e}")

def median_ms(fn, items, n=300):
    ts = []
    for it in items[:n]:
        t0 = time.perf_counter(); fn(it); ts.append(1e3 * (time.perf_counter() - t0))
    return float(np.median(ts))
t_full = median_ms(lambda o: brain_c.report(fz_c.transform([o])), te_obs)
t_edge = median_ms(lambda o: edge.decide(o.features, o.text), te_obs)
print(f"median decision time on this machine: scikit-learn {t_full:.2f} ms | NumPy runtime {t_edge:.3f} ms")

In [ ]:
# peak memory of a fresh process that loads only NumPy + the runtime and makes 300 decisions
json.dump([{"f": dict(o.features), "t": o.text} for o in te_obs[:300]], open("obs.json", "w"))
probe = '''
import json, importlib.util
spec = importlib.util.spec_from_file_location("rt", "conclave/edge/runtime.py")
rt = importlib.util.module_from_spec(spec); spec.loader.exec_module(rt)
agent = rt.EdgeAgent("conclave_agent.npz")
for o in json.load(open("obs.json")):
    agent.decide(o["f"], o["t"])
import sys
print([l for l in open("/proc/self/status") if l.startswith("VmHWM")][0].split()[1],
      "sklearn" in sys.modules, "scipy" in sys.modules)
'''
kb, sk, sp = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True).stdout.split()
print(f"peak RAM of the runtime process: {int(kb) / 1024:.1f} MB (scikit-learn loaded: {sk}, SciPy loaded: {sp})")
boards = pd.DataFrame([{"board": p.name, "RAM (MB)": p.ram_mb or "-",
                        "est. latency, runtime (ms)": round(t_edge / p.scalar_score, 2),
                        "est. latency, scikit-learn (ms)": round(t_full / p.scalar_score, 1)}
                       for k, p in PROFILES.items() if k != "host"]).set_index("board")
display(boards)
print("Board latencies are estimated by scaling this machine's latency with each board's relative single-core speed.")

## 6. Summary
* **Data.** Two public datasets (HumAID: 76,484 crisis tweets in 10 classes; BFCL v3: 1,252 tool-routing requests and 2,628 relevance cases) and a flood-response simulator that generates 6,400 labelled decisions from 160 expert episodes. The EDA shows strongly imbalanced classes and short texts with class-specific vocabulary.
* **Working process.** Hashed TF–IDF text features and standardised sensors feed a heterogeneous committee; the spread of its members' posteriors gives aleatoric and epistemic uncertainty; a calibrated gate decides whether to act, gather, abstain or escalate; a safety layer keeps irreversible actions under control.
* **Without internet.** The local committee keeps working when the network is down, while a cloud-dependent agent collapses (paper: return 33.5 vs −127.3 with 73% of steps offline).
* **Against other ML agents.** CONCLAVE gives the highest closed-loop return and the fewest casualties among nine ML-based agents (paper, 120 episodes, paired bootstrap).
* **Low-cost hardware.** The exported agent is about 128 KB; the NumPy-only runtime needs about 32 MB of RAM and an estimated 1.7 ms per decision on a Raspberry Pi Zero 2 W.

To reproduce every number of the paper at full scale, run `python scripts/run_all.py` in the repository (about one hour on a 2-core CPU).